# Classificação Fine-Grained de Raças de Cães

**Trabalho final — Visão Computacional**
Pós-graduação em LLM e IA Generativa

Autores: Victor Macaubas e Mari
Data: agosto/2026

Repositório: https://github.com/victormacaubas/project-image-processing

---

> **Este notebook é autossuficiente e roda em ~2 minutos.**
>
> Basta `Ambiente de execução → Executar tudo`. A primeira célula clona o
> repositório e instala o que falta; nenhum arquivo adicional é necessário e
> nada precisa ser baixado.
>
> Os resultados, gráficos e a análise de erros são reconstruídos a partir das
> predições (logits) versionadas no repositório — por isso é rápido.
>
> Para reexecutar os treinos do zero, troque `RETRAIN` para `True` na célula de
> setup. Aí sim leva ~2 horas com GPU.

In [ ]:
# ─── Bootstrap ──────────────────────────────────────────────────────────────
# Deixa o notebook autossuficiente: clona o repositório e instala o que falta.
# Idempotente — rodar duas vezes não causa dano.
#
# NÃO reinstala torch/torchvision: o Colab já os traz, e reinstalar dispara
# "restart runtime", que aborta a execução de ponta a ponta.

REPO_URL = "https://github.com/victormacaubas/project-image-processing.git"
REPO_NAME = "project-image-processing"

import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules


def _run(cmd: list[str]) -> None:
    """Roda o comando e, se falhar, mostra a saída real.

    Sem isso, um pip que falha com -q some silenciosamente e o erro só
    aparece 20 células depois, como ImportError sem contexto.
    """
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr, file=sys.stderr)
        raise RuntimeError(f"Falhou: {' '.join(cmd)}")


def _find_repo_root() -> Path:
    """Localiza a raiz do repo, clonando se necessário. Idempotente."""
    here = Path.cwd()

    # Já estamos dentro do repositório?
    for candidate in (here, *here.parents):
        if (candidate / "src" / "dogs" / "config.py").exists():
            return candidate

    # Já clonado num subdiretório?
    if (here / REPO_NAME / "src" / "dogs" / "config.py").exists():
        return here / REPO_NAME

    # Clonar.
    print(f"Clonando {REPO_URL} ...")
    _run(["git", "clone", "--depth", "1", REPO_URL, REPO_NAME])
    return here / REPO_NAME


REPO_ROOT = _find_repo_root()

if IN_COLAB:
    reqs = REPO_ROOT / "requirements-colab.txt"
    if reqs.exists():
        print("Instalando dependências ...")
        _run([sys.executable, "-m", "pip", "install", "-q", "-r", str(reqs)])

# Torna o pacote `dogs` importável, sem duplicar entradas em sys.path.
src = str(REPO_ROOT / "src")
if src not in sys.path:
    sys.path.insert(0, src)

print(f"Repositório: {REPO_ROOT}")

In [ ]:
# ─── Verificação ────────────────────────────────────────────────────────────
# Falha aqui, alto e claro, em vez de estourar um erro críptico 20 células
# adiante. Se esta célula passar, o notebook roda até o fim.

import importlib

problemas = []

for pacote in ["torch", "torchvision", "numpy", "pandas", "sklearn",
               "matplotlib", "seaborn", "datasets"]:
    try:
        importlib.import_module(pacote)
    except ImportError as erro:
        problemas.append(f"pacote ausente: {pacote} ({erro})")

try:
    from dogs.config import describe_environment, ensure_dirs
    ensure_dirs()
    print(describe_environment())
except Exception as erro:
    problemas.append(f"pacote `dogs` não importável: {erro}")

if problemas:
    raise RuntimeError(
        "Ambiente incompleto:\n  - " + "\n  - ".join(problemas)
        + "\n\nRode a célula de bootstrap acima antes desta."
    )

print("\nAmbiente OK.")

In [ ]:
# ─── Setup ──────────────────────────────────────────────────────────────────
# False -> reconstrói tudo das predições versionadas (~2 min). É o padrão.
# True  -> reexecuta os treinos a partir do dataset (~2 h com GPU).
RETRAIN = False

import logging
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt, seaborn as sns

from dogs.config import (TrainConfig, FEATURES_DIR, CHECKPOINT_DIR,
                         PREDICTIONS_DIR, RESULTS_CSV)

# force=True é obrigatório: o Colab instala um handler no logger raiz antes desta
# célula, e sem isso o basicConfig sai calado, o nível fica em WARNING e nenhum log
# de progresso do treino aparece.
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", force=True)
torch.manual_seed(42)
np.random.seed(42)

### Artefatos

Este notebook é **o documento de trabalho e a entrega ao mesmo tempo**. Durante a
semana ele fica incompleto; a célula abaixo mostra o que já existe e o que falta.

| Nível | Precisa de | Vem de | Tempo |
|---|---|---|---|
| **Padrão** (`RETRAIN = False`) | predições (~3 MB cada) | repositório | ~2 min |
| Reproduzir os treinos | embeddings + checkpoints | Drive | ~2 h |

No nível padrão nada é baixado: os logits de cada experimento estão versionados
no Git, e deles saem a tabela, os gráficos e a análise de erros. Um checkpoint de
ResNet50 tem ~100 MB; os logits que ele produz, ~3 MB — e para refazer a
*análise*, os logits bastam.

In [ ]:
# ─── Estado dos experimentos ────────────────────────────────────────────────
# Durante a semana isto é o painel de progresso. Na sexta, precisa estar todo OK.
DRIVE_FOLDER_ID = "12XC2LBi7NK02kTdIMN6FAU_Wf7sbKYZ8"

from pathlib import Path

# Cada entrada: (nome do experimento, split, quem faz)
ESPERADOS = [
    ("E1_scratch",        "val",  "Mari"),
    ("E2_linear_probe",   "val",  "Mari"),
    ("E3_finetune_last2", "val",  "Victor"),
    ("E3_finetune_last2", "test", "Victor"),   # só na sexta
]

ASSINATURAS = {".npy": b"\x93NUMPY", ".npz": b"PK", ".pt": b"PK"}


def arquivo_valido(caminho: Path) -> bool:
    """Detecta HTML gravado com extensão de dados.

    O Google intercepta downloads grandes com uma página de aviso de vírus; sem
    esta checagem o gdown grava esse HTML como .npy, e o erro só aparece muito
    depois como um "corrupt file" incompreensível.
    """
    esperado = ASSINATURAS.get(caminho.suffix)
    if esperado is None:
        return True
    with caminho.open("rb") as f:
        return f.read(len(esperado)) == esperado


def baixar_artefatos() -> None:
    import gdown
    destino = FEATURES_DIR.parent
    gdown.download_folder(id=DRIVE_FOLDER_ID, output=str(destino),
                          quiet=False, use_cookies=False)

    baixados = [p for p in destino.rglob("*") if p.is_file() and p.suffix in ASSINATURAS]
    if not baixados:
        raise RuntimeError(
            "O download do Drive não trouxe nenhum arquivo de dados.\n"
            "As subpastas são recriadas mesmo vazias, então isto passa silenciosamente "
            "e só estoura células adiante como FileNotFoundError.\n"
            "Confira: (1) os .npy estão mesmo na pasta do Drive; (2) o "
            "compartilhamento 'Qualquer pessoa com o link' vale para os arquivos, "
            "não apenas para a pasta."
        )

    corrompidos = [p for p in baixados if not arquivo_valido(p)]
    if corrompidos:
        for p in corrompidos:
            p.unlink()
        raise RuntimeError(
            f"{len(corrompidos)} arquivo(s) baixados como HTML, não como dados.\n"
            "Causa provável: a pasta do Drive não está pública, ou o Google "
            "interceptou o download (comum acima de 100 MB).\n"
            "Saídas: (1) conferir acesso 'Qualquer pessoa com o link'; "
            "(2) publicar numa GitHub Release; (3) rodar com RETRAIN = False."
        )


if RETRAIN and not (FEATURES_DIR / "resnet50_train_X.npy").exists():
    baixar_artefatos()

# ─── Painel ─────────────────────────────────────────────────────────────────
print(f"{'experimento':22s} {'split':6s} {'dono':7s} estado")
print("─" * 58)

faltando = []
for nome, split, dono in ESPERADOS:
    caminho = PREDICTIONS_DIR / f"{nome}_{split}.npz"
    if caminho.exists():
        estado = f"ok  ({caminho.stat().st_size / 1e6:.1f} MB)"
    else:
        estado = "FALTA"
        faltando.append(f"{nome}/{split}")
    print(f"{nome:22s} {split:6s} {dono:7s} {estado}")

print()
if faltando:
    print(f"Faltam {len(faltando)}: {', '.join(faltando)}")
    print("Normal durante a semana. Na sexta, tem que estar tudo ok.")
else:
    print("Todos os experimentos presentes — pronto para o teste de entrega.")

### Bastidor

Operações que usamos durante o desenvolvimento e que **não fazem parte da
entrega**: montar o Drive, gerar embeddings, subir artefatos.

Ficam atrás de `MODO_TRABALHO` em vez de serem apagadas — assim ninguém esquece
de removê-las na sexta, e quem for reproduzir o trabalho vê o que foi feito.

Com `MODO_TRABALHO = False` (padrão) esta célula não faz nada: quem estiver
corrigindo não recebe pedido de autorização do Google Drive.

In [ ]:
MODO_TRABALHO = False   # True só enquanto estamos desenvolvendo

if MODO_TRABALHO:
    from pathlib import Path

    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE = Path('/content/drive/MyDrive/project-image-processing')
    print("features/   ", (DRIVE / 'features').exists())
    print("checkpoints/", (DRIVE / 'checkpoints').exists())

    # ── Descomente conforme a tarefa do dia ──────────────────────────────

    # Teste de fumaça do data.py. Baixa 776 MB na primeira vez (5-10 min).
    # Esperado: [64, 3, 224, 224] · [64] · 120 · n02085620-Chihuahua
    #
    # from dogs.data import load_data
    # dados = load_data(TrainConfig(experiment_name="smoke"))
    # imagens, rotulos = next(iter(dados.train_loader))
    # print(imagens.shape, rotulos.shape, dados.num_classes)
    # print(dados.class_names[:3])

    # Gerar os embeddings (~8 min na T4, roda uma vez):
    # !python -m dogs.features --backbone resnet50

    # Subir para o Drive, e avisar a Mari:
    # !cp data/processed/features/*.npy "{DRIVE}/features/"
    # !ls -lh "{DRIVE}/features/"

---
# 1. Descrição do problema

<!-- OBRIGATÓRIO NA ENTREGA — dona: Mari, prazo: quinta -->

**Escrever aqui:**

- O que é classificação fine-grained e por que difere de classificação genérica
- Por que é difícil: baixa variância entre classes, alta variância dentro da classe
- Por que importa (aplicações reais)
- A pergunta que guia o trabalho: *quanto de representação visual precisa ser
  aprendido versus transferido, com dados limitados?*
- O que consideramos sucesso

---
# 2. Descrição da base de dados

<!-- OBRIGATÓRIO NA ENTREGA — dona: Mari, prazo: quinta -->

**Escrever aqui:** origem, licença, 20.580 imagens, 120 classes, split oficial
12.000/8.580, como separamos validação, distribuição de classes, resolução das imagens.

**Não esquecer:** Stanford Dogs é derivado do ImageNet. Backbones pré-treinados em
ImageNet já viram essas imagens. Discutir na seção 6.

In [ ]:
# EDA — colar do notebooks/01_eda.ipynb
# Distribuição de classes, grid de amostras, exemplos de raças visualmente próximas

---
# 3. Metodologia

<!-- OBRIGATÓRIO NA ENTREGA — donos: Mari e Victor, prazo: quinta -->

**Escrever aqui:**

- Estratégia geral: progressão do zero → transferido → adaptado
- Pré-processamento e augmentation (e por que só no treino)
- Arquiteturas de cada experimento
- Protocolo de treino: AdamW, cosine schedule, early stopping na val, label smoothing
- Métricas: top-1, top-5, F1 macro — e por que cada uma
- **Por que pré-computamos embeddings:** decisão de engenharia que viabilizou o prazo

## 3.1 Extração de embeddings

Passo executado uma vez, fora do notebook:

```bash
python -m dogs.features --backbone resnet50
```

Gera `{backbone}_{split}_{X,y}.npy` em `data/processed/features/`.

In [ ]:
# Os embeddings (~170 MB) não são versionados, e só o treino do linear probe precisa
# deles. Com RETRAIN = False o E2 sai dos logits salvos, como os outros experimentos.
if RETRAIN:
    from dogs.features import load_features

    X_train, y_train = load_features(split="train")
    X_val, y_val = load_features(split="val")
    print(X_train.shape, X_val.shape)
else:
    print("Embeddings não carregados (RETRAIN = False); o E2 vem dos logits salvos.")


---
# 4. Experimentos

<!-- OBRIGATÓRIO NA ENTREGA -->

Cada experimento: o que testa, como foi configurado, o que aconteceu.

## E1 — CNN treinada do zero

**Hipótese:** sem transferência, 120 classes fine-grained com ~100 imagens de treino
por classe não dão sinal suficiente. Esperamos acurácia baixa.

*Dona: Mari*

In [ ]:
from dogs.evaluate import load_predictions, metrics_from_logits
from dogs.models import SmallCNN  # noqa: F401  (Mari implementa e treina aqui)

NOME = "E1_scratch"

# Sem o modelo implementado não há o que treinar; o experimento aparece como
# pendente em vez de interromper a execução de ponta a ponta.
try:
    logits, labels = load_predictions(NOME, "val")
    print(f"val:  {metrics_from_logits(logits, labels)}")
except FileNotFoundError:
    print(f"{NOME}: predições ainda não geradas.")


## E2 — Linear probe sobre backbone congelado

**Hipótese:** a representação do ImageNet já separa bem as raças, mesmo sem nenhuma
adaptação — um classificador linear deve superar E1 por larga margem.

*Dona: Mari*

In [ ]:
from dogs.evaluate import load_predictions, metrics_from_logits
from dogs.models import LinearProbe  # noqa: F401  (Mari implementa e treina aqui)

NOME = "E2_linear_probe"

# Sem o modelo implementado não há o que treinar; o experimento aparece como
# pendente em vez de interromper a execução de ponta a ponta.
try:
    logits, labels = load_predictions(NOME, "val")
    print(f"val:  {metrics_from_logits(logits, labels)}")
except FileNotFoundError:
    print(f"{NOME}: predições ainda não geradas.")


## E3 — Fine-tuning parcial

**Hipótese:** descongelar os blocos finais permite adaptar as features de alto nível
ao domínio e deve superar E2.

*Dono: Victor*

In [ ]:
from dogs.config import TrainConfig
from dogs.data import _load_raw, load_data
from dogs.evaluate import (
    load_predictions,
    log_result,
    metrics_from_logits,
    predict,
    save_predictions,
)
from dogs.models import build_finetune_model
from dogs.train import get_device, load_checkpoint, train_model

config = TrainConfig(
    experiment_name="E3_finetune_last2",
    unfreeze_last_n_blocks=2,
    learning_rate=1e-4,
    num_epochs=15,
)

if RETRAIN:
    data = load_data(config)
    class_names = data.class_names

    model = build_finetune_model("resnet50", config.unfreeze_last_n_blocks)
    train_model(model, data.train_loader, data.val_loader, config)
    model = load_checkpoint(model, config)

    device = get_device()
    logits_val, labels_val = predict(model, data.val_loader, device)
    logits_test, labels_test = predict(model, data.test_loader, device)
    metrics_val = metrics_from_logits(logits_val, labels_val)
    metrics_test = metrics_from_logits(logits_test, labels_test)

    for split, logits, labels, metrics in [
        ("val", logits_val, labels_val, metrics_val),
        ("test", logits_test, labels_test, metrics_test),
    ]:
        log_result(config.experiment_name, split, metrics)
        save_predictions(config.experiment_name, split, logits, labels)
else:
    class_names = list(_load_raw()["train"].features["label"].names)

    logits_val, labels_val = load_predictions(config.experiment_name, "val")
    logits_test, labels_test = load_predictions(config.experiment_name, "test")
    metrics_val = metrics_from_logits(logits_val, labels_val)
    metrics_test = metrics_from_logits(logits_test, labels_test)

    # Checkpoint (~100 MB) nao e versionado no repo; carrega so se existir localmente.
    if config.checkpoint_path().exists():
        model = build_finetune_model("resnet50", config.unfreeze_last_n_blocks)
        model = load_checkpoint(model, config)

print(f"val:  {metrics_val}")
print(f"test: {metrics_test}")

---
# 5. Resultados

<!-- OBRIGATÓRIO NA ENTREGA -->

In [ ]:
results = pd.read_csv(RESULTS_CSV)
results.sort_values("top1", ascending=False)

In [ ]:
from dogs.config import FIGURES_DIR
from dogs.viz import comparar_experimentos

comparar_experimentos(results, salvar_em=FIGURES_DIR / "comparacao_top1.png")

---
# 6. Análise

<!-- OBRIGATÓRIO NA ENTREGA — dono: Victor, prazo: quinta -->

## 6.1 O salto do transfer learning

Comparar E1 vs. E2 vs. E3 e interpretar a magnitude da diferença.

## 6.2 Quais raças o modelo confunde

Pares mais confundidos. As confusões são visualmente plausíveis? Um humano erraria
os mesmos casos?

## 6.3 ⚠️ Contaminação entre Stanford Dogs e ImageNet

Stanford Dogs foi construído a partir do ImageNet. O backbone pré-treinado em
ImageNet-1k **já viu essas imagens**. Nossos números de transfer learning são,
portanto, otimistas e não estimam o desempenho em um domínio novo.

Discutir: o que isso invalida, o que continua válido, e como um experimento futuro
poderia medir o efeito (ex.: avaliar em fotos de cães fora do ImageNet).

## 6.4 Limitações

Orçamento computacional, ausência de busca de hiperparâmetros, execução única
por experimento (sem barras de erro), split de teste tocado uma só vez.

In [ ]:
from dogs.config import FIGURES_DIR
from dogs.data import _load_raw
from dogs.evaluate import most_confused_pairs
from dogs.viz import (
    grid_pares_confundidos,
    grid_piores_erros,
    matriz_confusao_recorte,
    nome_legivel,
)

pares = most_confused_pairs(logits_test, labels_test, class_names, top_n=10)
for real, previsto, contagem in pares:
    print(f"{nome_legivel(real):25s} -> {nome_legivel(previsto):25s} {contagem}x")

test_dataset = _load_raw()["test"]

grid_pares_confundidos(
    test_dataset,
    logits_test,
    labels_test,
    class_names,
    top_n=5,
    salvar_em=FIGURES_DIR / "pares_confundidos.png",
)

# Recorte a partir dos 5 pares mais frequentes: 10 pares já renderiam uma matriz de
# ~20 classes, tão ilegível quanto a 120x120 que o recorte existe para evitar.
nomes_unicos = list(
    dict.fromkeys(nome for real, previsto, _ in pares[:5] for nome in (real, previsto))
)
indices_classes = [class_names.index(nome) for nome in nomes_unicos]

matriz_confusao_recorte(
    logits_test,
    labels_test,
    class_names,
    indices_classes,
    salvar_em=FIGURES_DIR / "matriz_confusao_recorte.png",
)

grid_piores_erros(
    test_dataset,
    logits_test,
    labels_test,
    class_names,
    top_n=10,
    salvar_em=FIGURES_DIR / "piores_erros.png",
)

---
# 7. Conclusões

<!-- OBRIGATÓRIO NA ENTREGA — dona: Mari, prazo: sexta -->

**Escrever aqui:**

- Resposta direta à pergunta da seção 1
- O que os números mostraram, incluindo o que surpreendeu
- O que faríamos com mais tempo (E5, E6, métodos fine-grained com atenção por partes)

---
## Referências

- Khosla et al. (2011). *Novel Dataset for Fine-Grained Image Categorization: Stanford Dogs.*
- He et al. (2016). *Deep Residual Learning for Image Recognition.*
- Radford et al. (2021). *Learning Transferable Visual Models From Natural Language Supervision.*